# Classic init (`all_init`) — accuracy vs finetune learning rate

How the **classic** (`all_init`, own-class-mean) transfer performs across finetune learning rates
on the synthetic DLCC test cases, for protograph variants **p1 / p2 / p3**.

All runs use the **same `synthetic_compare` recipe** (codes normalized to 8 and mirrored into
`syn1neg`, dim 200, 100 walks, depth 3, 5+5 epochs, seed 42); only the fine-tune rate differs.
Numbers are loaded directly from the result files:

| lr | source |
|----|--------|
| 0.025 (from-scratch) | `notebooks/synthetic_compare/{p1,p2,p3}_classic_lr025.json` (recipe-consistent reruns) |
| 0.0025 (protected) | `notebooks/synthetic_compare/results.json` (p1/p2, vanilla) + `notebooks/p3_classic/results.json` (p3) |
| 0.0001 | `output/synthetic/protograph/..._finetune_lr_0001_...` (partial: tc01/tc07, p1/p2 only) |
| 0.001 | **not run** — no such directory exists |

> The 0.025 column matches the `p1_classic`/`p2_classic` values in the thesis main table
> `tab:synthetic_final` exactly; `p3_classic`@0.025 (0.770) agrees with the synthetic-benchmark
> run (0.774) to within run-to-run noise.
>
> ⚠️ Only **lr 0.0025 vs 0.025 over tc01–tc15** is a clean apples-to-apples sweep.
> The lr 0.0001 column covers only tc01/tc07 (and only p1/p2), so it is indicative, not comparable.


In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

# Notebook lives in notebooks/ -> walk up to the repo root.
REPO = Path.cwd()
while not (REPO / "output" / "synthetic_benchmark" / "summary.json").exists() and REPO != REPO.parent:
    REPO = REPO.parent

TCS = [f"tc{i:02d}" for i in range(1, 16)]   # tc01 .. tc15 (comparable scope)
PROTOS = ["p1", "p2", "p3"]
print("repo root:", REPO)


## Load results

In [ ]:
# --- lr 0.025 (from-scratch): recipe-consistent reruns, flat {tc: acc} ---
acc_025 = {p: json.loads((REPO / f"notebooks/synthetic_compare/{p}_classic_lr025.json").read_text())
           for p in PROTOS}

# --- lr 0.0025 (protected) + vanilla (always from-scratch 0.025) ---
res = json.loads((REPO / "notebooks/synthetic_compare/results.json").read_text())
acc_0025 = {p: {} for p in PROTOS}
vanilla_025 = {}
for tc, variants in res.items():
    for v in variants:
        if v["variant"] == "vanilla":
            vanilla_025[tc] = v["final_acc"]
        for p in ("p1", "p2"):
            if v["variant"] == f"{p}_classic":
                acc_0025[p][tc] = v["final_acc"]

# --- lr 0.0025: p3 (dict tc -> single variant dict) ---
p3res = json.loads((REPO / "notebooks/p3_classic/results.json").read_text())
for tc, v in p3res.items():
    acc_0025["p3"][tc] = v["final_acc"]

# --- lr 0.0001: partial reference (anchor_reg=0.01, tc01/tc07, p1/p2 only) ---
acc_0001_partial = {
    "p1": {"tc01": 0.6000, "tc07": 0.4850},
    "p2": {"tc01": 0.5850, "tc07": 0.4775},
}

print("loaded tcs @0.025 :", {p: len(acc_025[p]) for p in PROTOS})
print("loaded tcs @0.0025:", {p: len(acc_0025[p]) for p in PROTOS})


## Run-to-run fluctuation (noise floor)

To judge whether a difference is meaningful we first estimate seed/run noise. The norm-ablation
runs (`output/synthetic_norm_ablation/`) contain `*_classic_raw` and `*_classic_unit`, which differ
only by an **inert** post-hoc normalization (classic normalization is a no-op) — so the pair is two
independent finetune runs of the *same* config, and their difference estimates noise.

In [ ]:
abl = json.loads((REPO / "output/synthetic_norm_ablation/summary.json").read_text())
acc_abl = {(r["tc"], r["condition"]): r["final_acc"] for r in abl}
abl_tcs = sorted({t for (t, c) in acc_abl})
diffs = [acc_abl[(tc, f"{p}_classic_raw")] - acc_abl[(tc, f"{p}_classic_unit")]
         for p in PROTOS for tc in abl_tcs
         if (tc, f"{p}_classic_raw") in acc_abl and (tc, f"{p}_classic_unit") in acc_abl]
diffs = np.array(diffs)
std_per_tc   = np.sqrt((diffs**2).mean()) / np.sqrt(2)     # per single (proto, tc) run
std_of_mean  = std_per_tc / np.sqrt(len(TCS))              # std of a 15-tc mean
print(f"paired runs        : {len(diffs)}")
print(f"std per test case  : {std_per_tc:.4f}  (1 sigma; one tc can swing this much from noise)")
print(f"std of 15-tc mean  : {std_of_mean:.4f}  -> treat mean diffs < ~{2*std_of_mean:.3f} as noise")


## Summary table — mean accuracy across test cases

In [ ]:
def mean_over(d, tcs=TCS):
    vals = [d[t] for t in tcs if t in d]
    return np.mean(vals) if vals else np.nan

summary_tbl = pd.DataFrame({
    "lr 0.0001*": {p: (np.mean(list(acc_0001_partial[p].values()))
                       if p in acc_0001_partial else np.nan) for p in PROTOS},
    "lr 0.001 (not run)": {p: np.nan for p in PROTOS},
    "lr 0.0025": {p: mean_over(acc_0025[p]) for p in PROTOS},
    "lr 0.025 (default)":  {p: mean_over(acc_025[p])  for p in PROTOS},
})
summary_tbl.index = [f"{p}_classic" for p in PROTOS]
# delta of LOWERING the lr from the default 0.025 down to 0.0025 (negative = worse)
summary_tbl["d(0.0025 - default)"] = summary_tbl["lr 0.0025"] - summary_tbl["lr 0.025 (default)"]
summary_tbl.loc["vanilla (lr 0.025)"] = [np.nan, np.nan, np.nan, mean_over(vanilla_025), np.nan]

# *lr 0.0001 = mean over tc01/tc07 only (p1/p2) -> not comparable to the 15-tc means.
summary_tbl.round(4)


## Analysis — did lowering the learning rate (0.025 → 0.0025) help?

**No.** Against a noise floor of ≈0.003 on the 15-test-case mean (so differences below ~0.007 are
not meaningful), reducing the finetune learning rate from the from-scratch **0.025** down to the
protected **0.0025** ranged from *no effect* to a *large collapse* — it never improved classic init:

| variant | 0.025 (from-scratch) | 0.0025 (protected) | Δ | verdict |
|---|---|---|---|---|
| **p2_classic** | 0.761 | 0.759 | **−0.002** | within noise — effectively unchanged |
| **p3_classic** | 0.770 | 0.741 | **−0.029** | several × the mean-noise — small but real loss |
| **p1_classic** | 0.750 | 0.609 | **−0.141** | far outside noise — a large collapse |

The **p1 collapse** is not uniform: it comes from a handful of test cases (e.g. tc03, tc06, tc09–tc11)
where p1 at the low lr sits at **chance (~0.50)**. The high-norm class-mean init (codes at norm 8,
mirrored into `syn1neg`) plus the small step size saturates the SGNS sigmoids, so training barely
moves the embeddings (cos(epoch0, epoch5) ≈ 0.995) — the same *protection* that the concept-bound
construction needs, but harmful here because the coarse class-mean init must move to solve the tasks.

**Conclusion:** the from-scratch rate 0.025 is the better setting for the classic transfer. Lowering
it to the protected 0.0025 leaves p2 unchanged, modestly hurts p3, and freezes/under-trains p1. The
protected low rate is a property the concept-bound init *requires*, not a universally safe choice.


## Detailed per-test-case breakdown (lr 0.0025 vs 0.025)

In [ ]:
detail = pd.DataFrame(
    [{
        "tc": tc,
        "p1@0.0025": acc_0025["p1"].get(tc), "p1@0.025": acc_025["p1"].get(tc),
        "p2@0.0025": acc_0025["p2"].get(tc), "p2@0.025": acc_025["p2"].get(tc),
        "p3@0.0025": acc_0025["p3"].get(tc), "p3@0.025": acc_025["p3"].get(tc),
    } for tc in TCS]
).set_index("tc")
detail.loc["MEAN"] = detail.mean()
detail.round(4)


## Plot

In [ ]:
import matplotlib.pyplot as plt

plot_df = summary_tbl.loc[[f"{p}_classic" for p in PROTOS], ["lr 0.0025", "lr 0.025 (default)"]]
ax = plot_df.plot(kind="bar", figsize=(7, 4), rot=0)
ax.axhline(mean_over(vanilla_025), ls="--", color="gray", lw=1.5,
           label="vanilla baseline (lr 0.025)")
ax.set_ylabel("mean accuracy (tc01-tc15)")
ax.set_ylim(0.45, 0.85)
ax.set_title("Classic init: mean accuracy by finetune learning rate")
ax.legend()
plt.tight_layout()
plt.show()


## Takeaways

- **Higher finetune lr is better for classic** across all three protographs (monotonic 0.0001 → 0.0025 → 0.025).
- **p1 is the lr-sensitive one**: it collapses to ~0.61 at lr 0.0025 while p2/p3 hold ~0.74–0.76; p1 only recovers at lr 0.025.
- **At lr 0.025 all three sit at/above the vanilla no-pretrain baseline (~0.735)**, p3 best (~0.770).
- **Recipe-consistent**: the 0.025 column is from the `synthetic_compare` reruns, so p1/p2 match the thesis
  main table exactly and p3@0.025 (0.770) is a true same-recipe number (≈ benchmark 0.774, within noise).
- **Gaps**: lr 0.001 was never run; p3 has no lr 0.0001 run; the lr 0.0001 runs cover only tc01/tc07.
- All numbers are read directly from the result files — none are hand-entered except the partial lr 0.0001 reference.
